# MoSim artifact guide

This notebook builds MoSim, reproduces Tables II–IV, inspects the prepared inputs, and runs one small bandwidth-sensitivity experiment. It follows the repository scripts instead of duplicating their preprocessing logic.

Run the cells from top to bottom. The bundled reproduction requires no GPU.

## Setup

Requirements: Rust 1.78+, Python 3.9+, and Linux or macOS. The reproduction scripts use only the Python standard library; Jupyter is required only for this notebook.

In [ ]:
import csv
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'simulator-trace-timer-bw.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'simulator-trace-timer-bw.py').exists(), 'Run this notebook from the MoSim repository.'
os.chdir(ROOT)

cargo_bin = Path.home() / '.cargo' / 'bin'
path_value = os.environ.get('PATH', '')
if cargo_bin.exists() and str(cargo_bin) not in path_value.split(os.pathsep):
    os.environ['PATH'] = f'{cargo_bin}{os.pathsep}{path_value}'

def sh(command, *, quiet=False, **kwargs):
    completed = subprocess.run(
        command, cwd=ROOT, capture_output=True, text=True, **kwargs
    )
    if completed.returncode != 0:
        raise RuntimeError(
            f'command failed ({completed.returncode}): {command}\n'
            f'stdout:\n{completed.stdout}\nstderr:\n{completed.stderr}'
        )
    if not quiet:
        if completed.stdout:
            print(completed.stdout.rstrip())
        if completed.stderr:
            print(completed.stderr.rstrip(), file=sys.stderr)
    return completed

SCRATCH = Path(tempfile.mkdtemp(prefix='mosim-guide-'))

print('repository:', ROOT)
print('python:    ', sys.version.split()[0])
for tool in ('cargo', 'rustc'):
    executable = shutil.which(tool)
    version = subprocess.run([tool, '--version'], capture_output=True, text=True).stdout.strip() if executable else 'not found'
    print(f'{tool:10}', version)

In [ ]:
assert shutil.which('cargo'), 'Install Rust from https://rustup.rs and restart the kernel.'
started = time.time()
sh(['cargo', 'build', '--release', '--locked', '--manifest-path', 'mosim/Cargo.toml'], quiet=True)
binary = ROOT / 'mosim/target/release/mosim'
assert binary.exists()
print(f'built {binary.relative_to(ROOT)} in {time.time() - started:.1f}s')

## Reproduce Tables II–IV

`scripts/run.sh` prepares the bundled inputs and runs the paper configurations. `scripts/analyze.sh` generates the three tables. `FORCE=1` and `FORCE_PREPARE=1` ensure that changed parameters or inputs do not reuse cached outputs.

In [ ]:
run_env = os.environ.copy()
run_env.update(
    RESULT_ROOT='result',
    TRACE_NAMES='32gpu 16gpu',
    TRACE='0',
    BW_MBPS='463',
    INTRA_BW='16384',
    NUM_NODE='4',
    NUM_NODE_32GPU='4',
    NUM_NODE_16GPU='2',
    NUM_GPU_PER_NODE='8',
    NUM_CPU_PER_NODE='256',
    SINGLE_JOB_PROFILES='astrasim profiling testbed',
    SCHEDULES='colocate k8s-bin-packing k8s-load-balancing',
    INFERENCE_MODELS='none fixed fixed-10 mosim',
    RAWDATA='result/rawdata.csv',
    OUTDIR='result/paper',
    FORCE='1',
    FORCE_PREPARE='1',
)
started = time.time()
sh(['bash', 'scripts/run.sh'], quiet=True, env=run_env)
sh(['bash', 'scripts/analyze.sh'], quiet=True, env=run_env)
print(f'reproduction completed in {time.time() - started:.1f}s')

for table_number in (2, 3, 4):
    table_path = ROOT / f'result/paper/tab{table_number}/out/table{table_number}.md'
    print()
    print(table_path.read_text(encoding='utf-8').strip())

### Verify the expected artifact values

The check below compares all 50 generated table cells with the values expected after the load-balancing capacity fix.

In [ ]:
EXPECTED = {
    2: {
        'Average training time': [58.71, 56.92],
        'Average JCT': [73.64, 72.23],
        'P99 JCT': [67.19, 65.31],
        'Makespan': [47.51, 46.42],
    },
    3: {
        'Tiresias': [38.19, 36.38, 34.98, 50.42, 53.45, 56.75, 53.93, 54.20, 65.28, 65.44],
        'Pollux': [35.30, 33.00, 31.83, 48.46, 51.08, 54.72, 51.42, 51.86, 63.89, 63.64],
        'MoSim': [19.74, 11.10, 1.62, 24.84, 17.61, 17.05, 21.83, 19.60, 8.38, 7.72],
    },
    4: {
        'Tiresias': [0.32, 3499, 0.38, 7082],
        'Pollux': [0.27, 3184, 0.37, 6753],
        'MoSim': [0.13, 1579, 0.22, 2864],
    },
}

def load_table(table_number):
    path = ROOT / f'result/paper/tab{table_number}/out/table{table_number}.csv'
    with path.open(newline='', encoding='utf-8') as handle:
        rows = list(csv.reader(handle))
    return {row[0]: [float(value) for value in row[1:]] for row in rows[1:]}

mismatches = []
checked = 0
for table_number, expected_rows in EXPECTED.items():
    actual_rows = load_table(table_number)
    for row_name, expected_values in expected_rows.items():
        actual_values = actual_rows.get(row_name)
        if actual_values is None:
            mismatches.append(f'Table {table_number}: missing row {row_name}')
            continue
        if len(actual_values) != len(expected_values):
            mismatches.append(
                f'Table {table_number}, {row_name}: {len(actual_values)} values; expected {len(expected_values)}'
            )
            continue
        for column, (actual, expected) in enumerate(zip(actual_values, expected_values)):
            checked += 1
            if abs(actual - expected) >= 0.005:
                mismatches.append(
                    f'Table {table_number}, {row_name}, column {column}: {actual} != {expected}'
                )

assert not mismatches, '\n'.join(mismatches)
assert checked == 50, f'verified {checked} cells; expected 50'
print(f'verified {checked} table cells')

In [ ]:
gpu_util_path = ROOT / 'result/astrasim/runs/mosim/k8s-load-balancing-gpu_util.csv'
with gpu_util_path.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
    node_columns = [name for name in (reader.fieldnames or []) if name.startswith('node') and name.endswith('_avg_util')]
peak_node_util = max(float(row[column]) for row in rows for column in node_columns)
assert peak_node_util <= 100.0 + 1e-9
print(f'peak per-server GPU utilization: {peak_node_util:.1f}%')

## Evaluated configurations

The method names in the paper correspond to the configurations below. The repository does not vendor the original Tiresias or Pollux implementations.

| Method | Single-job profile | Interference model |
| --- | --- | --- |
| Tiresias | real-GPU profiling | none |
| Pollux | real-GPU profiling | fixed 10% networking-time penalty while sharing a server |
| **MoSim** | ASTRA-sim | `mosim` |

The artifact includes the prepared inputs and scripts for Tables II–IV. It does not include the ASTRA-sim profile-generation workflow or the source measurements for Fig. 2, Fig. 4, and Table V.

## Prepared inputs

Each simulation consumes a job trace, an iteration-time profile, and an aggregate network-demand profile. The raw files under `data/` require the preprocessing performed by `scripts/run.sh`; the prepared files are written under `result/<profile>/_build/`.

In [ ]:
PREPARED = ROOT / 'result/astrasim/_build'
INPUTS = {
    'job trace': PREPARED / 'testbed-trace_duration-derived.csv',
    'iteration profile': PREPARED / 'itertime_simulated-ar-v100-8gpu-ar-loading-time_prepared.csv',
    'network profile': PREPARED / 'network_chakra_ar_v100_8gpu_network_summary_prepared.csv',
}

for label, path in INPUTS.items():
    assert path.exists(), f'missing prepared input: {path}'
    with path.open(newline='', encoding='utf-8-sig') as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)
    print(f'{label:18} {path.relative_to(ROOT)}')
    print(f'  rows={len(rows)}, columns={reader.fieldnames}')

## Run one configuration

The helper below invokes the public Python CLI with prepared 32-GPU inputs. It removes any previous files for the same tag because direct CLI output is append-oriented.

In [ ]:
def run_one(*, tag, bandwidth=463, schedule='k8s-load-balancing'):
    log_path = SCRATCH / f'{tag}-timer.txt'
    csv_path = log_path.with_suffix('.csv')
    allocation_path = SCRATCH / f'{tag}-allocation.log'
    for path in (log_path, csv_path, allocation_path):
        path.unlink(missing_ok=True)

    command = [
        sys.executable, 'simulator-trace-timer-bw.py',
        '--num_node', '4',
        '--num_gpus_per_node', '8',
        '--num_cpus_per_node', '256',
        '--schedule', schedule,
        '--interference-model', 'mosim',
        '--jobtrace', str(INPUTS['job trace']),
        '--iteration_time_csv_file', str(INPUTS['iteration profile']),
        '--communication_volume_csv_file', str(INPUTS['network profile']),
        '--bandwidth', str(bandwidth),
        '--intra_bandwidth', '16384',
        '--allocationlog', str(allocation_path),
        '--log', str(log_path),
    ]
    sh(command, quiet=True)

    metrics = {}
    for line in log_path.read_text(encoding='utf-8').splitlines():
        if ':' not in line:
            continue
        key, value = line.split(':', 1)
        try:
            metrics[key.strip()] = float(value)
        except ValueError:
            pass
    return metrics

single_run = run_one(tag='single')
for metric in ('avg_jct', 'avg_training', 'Makespan', 'sim_runtime_seconds'):
    print(f'{metric:22} {single_run[metric]:,.3f}')

## Contention model in the current implementation

The paper-facing `mosim` name maps to the phase-aware `comms-iter-intra` implementation. Jobs progress through loading, compute, and communication events. Compute-phase jobs contribute no inter-server demand; communication-phase jobs share each server NIC in proportion to their demand, and the resulting factor scales their networking time.

| Concern | Implementation |
| --- | --- |
| Job demand | `mosim/src/gpu_job.rs` → `required_bandwidth` |
| Placement and NIC sharing | `mosim/src/gpu_cluster.rs` |
| Phase transitions and completion | `mosim/src/gpu_scheduler.rs` |
| Event queue | `mosim/src/timer.rs` |

The repository source is the authoritative specification for implementation details and optional contention-model extensions.

## Bandwidth sensitivity

This small sweep uses the same prepared workload while changing only the inter-server NIC bandwidth.

In [ ]:
print(f"{'bandwidth (MB/s)':>18} {'average JCT (s)':>18} {'makespan (s)':>16}")
for bandwidth in (100, 463, 1000):
    metrics = run_one(tag=f'bandwidth-{bandwidth}', bandwidth=bandwidth)
    print(f"{bandwidth:>18} {metrics['avg_jct']:>18,.1f} {metrics['Makespan']:>16,.1f}")

## Usage notes

- Use `scripts/run.sh` for the paper configuration; the standalone CLI defaults are not the paper settings.
- Set `FORCE=1` when changing experimental parameters and `FORCE_PREPARE=1` when changing source inputs.
- `TRACE=1 bash scripts/run.sh` enables detailed event logs and produces substantially larger output.
- The CLI consumes prepared CSV files. Generation of profiles for new models is outside this artifact.
- See `mosim/src/config.rs` and `python3 simulator-trace-timer-bw.py --help` for available options.

In [ ]:
shutil.rmtree(SCRATCH, ignore_errors=True)
print('removed notebook scratch output; result/ is retained')